In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree


# My Decision Tree Model

In [5]:
data = pd.read_csv("clean_data.csv")
test = pd.read_csv('clean_test.csv')


One hot encoding for the data set. We want to look at contract length and gender

In [6]:
df_encoded = pd.get_dummies(data, columns=['Contract Length','Gender'], prefix=['Contract Length','Gender'], dtype=int)


In [7]:
X = df_encoded[['Total Spend','Support Calls','Contract Length_Monthly','Last Interaction','Age','Contract Length_Quarterly','Gender_Female']]
y = df_encoded['Churn']
X_train, X_validation, y_train, y_validation = train_test_split(X,y, test_size=.3, random_state=123)



In [10]:
depth_limit = 10

#predict only using one feature
model = DecisionTreeClassifier(criterion = 'entropy', max_depth = depth_limit)
model.fit(X_train,y_train)

#predict on train data
y_pred_train = model.predict(X_train)

#predict on test data
y_pred_test = model.predict_proba(X_validation)

#if I wanted to visualize the decision tree

#plt.figure(figsize=(10,10))
#plot_tree(model,feature_names=features,class_names=['Not','Churned'],filled=True)
#plt.title(f'Decision tree(features:{features},Max Depth: {depth_limit})')

In [11]:
#look at the feature importances in order to determine what is deffecting the accuracy the most

model.feature_importances_

array([0.27651062, 0.36088047, 0.15718089, 0.05610321, 0.13540518,
       0.00045625, 0.01346338])

How accuracte is my decision tree model? How well does it work?

In [12]:
train_accuracy = metrics.accuracy_score(y_train,y_pred_train)
print(train_accuracy)


0.8916752283374021


In [13]:
#test_accuracy = metrics.accuracy_score(y_validation,y_pred_test)
from sklearn.metrics import roc_auc_score
churn_prob = y_pred_test[:,1]

print(roc_auc_score(y_validation,churn_prob))

0.9190885488321086


Now we want to run our predicitons for our test data based on the model that we trained. We first do one hot encoding for the test data, then with the same columns we created model based on we predicted whether a customer would churn or not churn. We then take our class predictions and put them into a csv file

In [14]:
test_encoded = pd.get_dummies(test, columns=['Contract Length','Gender'], prefix=['Contract Length','Gender'], dtype=int)


In [15]:
prob = test_encoded[["CustomerID"]].copy()
X_test=test_encoded[['Total Spend','Support Calls','Contract Length_Monthly','Last Interaction','Age','Contract Length_Quarterly','Gender_Female']]

class_prediction = model.predict_proba(X_test)
prob['Churn'] = class_prediction[:,1]


In [16]:
prob.to_csv("prob_dt_attempt4.csv",index=False)